In [ ]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
import pyvinecopulib as pv
from scipy.stats import genpareto, kendalltau, rankdata
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple, Literal
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from sklearn.covariance import LedoitWolf

In [ ]:
@dataclass
class Universe:
  Bonds:List[str]
  ManagedFutures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Low_Beta:List[str]

In [165]:
@dataclass
class ModelParams:
  distance_method:str="kendalltau"
  optimize:bool=True
  cvar_q:float|None=0.95
  use_lw_shrinkage:bool=False
  k_max:int=8
  n_sims:int=100
  return_scale:int=100
  shrink_target_ess:int=100
  shrink:bool=True
  min_effective_sample:int=30
  tail_power:float=2.0
  tail_threshold:float=0.80

  def __post_init__(self):
    allowed_methods = {"kendalltau", "pearsoncorr"}
    if self.distance_method not in allowed_methods:
      raise ValueError(f"Invalid status. Choose from {allowed_methods}")

    if not (0 <= self.cvar_q <= 1):
      raise ValueError("Quantile must be between 0 and 1")

In [166]:
@dataclass
class ClusterParams:
  quantiles: dict[str, float] = field(default_factory=dict)
  paths: np.ndarray = None

In [167]:
@dataclass
class SimParams:
  scale_factor:int=1000
  q_upper:float=0.95
  q_lower:float=0.95
  n_paths:int = 10000
  n_steps:int = 100
  lags:int = 10

In [168]:
@dataclass
class TailFit:
  c_L: float
  scale_L: float
  c_U: float
  scale_U: float
  p_l: float
  p_u: float
  u_lower: float
  u_upper: float
  ecdf_data: np.ndarray

In [169]:
@dataclass
class GARCHEVTCOPULAFit:
  vol_init:dict = field(default_factory=dict)
  r_init:dict = field(default_factory=dict)
  best_models:dict = field(default_factory=dict)
  best_fits:dict = field(default_factory=dict)
  best_lags:dict = field(default_factory=dict)
  marginal_distributions:dict = field(default_factory=dict)

In [215]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers_raw = list(asdict(universe).values())
    tmp = []
    for t in tickers_raw:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      if benchmark not in tickers_clean:
        tickers_clean.append(benchmark)

      df = yf.download(tickers_clean, start, end, interval)["Close"]

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    returns = data_raw.pct_change().dropna()
    self.universe = data_raw.columns

    return returns, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [216]:
class MarginalTailModel:
  def __init__(self, sim_params, ticker):
    self.dist_p = sim_params
    self.ticker = ticker
    self.tail_fit = None

  def fit_dist(self,r):
    full_idx = r.index
    r = r.dropna()
    valid_idx = r.index
    r = r.to_numpy()

    q_upper = self.dist_p.q_upper * 100
    q_lower = (1 - self.dist_p.q_lower) * 100

    u_upper = np.percentile(r, q_upper)
    u_lower = np.percentile(r, q_lower)

    upper_tail = r[r > u_upper]
    lower_tail = r[r < u_lower]

    c_U, _, scale_U = genpareto.fit(upper_tail - u_upper, floc=0)
    c_L, _, scale_L = genpareto.fit(u_lower - lower_tail, floc=0)

    p_l = np.mean(r < u_lower)
    p_u = np.mean(r > u_upper)

    lower_mask= r < u_lower
    upper_mask = r > u_upper
    body_mask = (~lower_mask) & (~upper_mask)

    u_resid = np.zeros_like(r, dtype=float)

    if np.any(lower_mask):
      cdf_l = 1 - genpareto.cdf(u_lower - r[lower_mask], c_L, scale=scale_L)
      u_resid[lower_mask] = p_l * cdf_l

    if np.any(body_mask):
      body = r[body_mask]
      ranks = pd.Series(body).rank(method='average').to_numpy()
      cdf_b = p_l + (1 - p_l - p_u) * (ranks / (len(body) + 1))
      u_resid[body_mask] = cdf_b

    if np.any(upper_mask):
      cdf_u = (1 - p_u) + p_u * genpareto.cdf(r[upper_mask] - u_upper, c_U, scale=scale_U)
      u_resid[upper_mask] = cdf_u


    self.tail_fit = TailFit(
      c_L=c_L,
      scale_L=scale_L,
      c_U=c_U,
      scale_U=scale_U,
      p_l=p_l,
      p_u=p_u,
      u_lower=u_lower,
      u_upper=u_upper,
      ecdf_data=r[(r >= u_lower) & (r <= u_upper)]
    )

    u_resid_full = pd.Series(np.nan, index=full_idx)
    u_resid_full.loc[valid_idx] = u_resid
    return u_resid_full

  def inverse_cdf(self, u_resid):
    z_resid_final = np.zeros_like(u_resid, dtype=float)

    for i in range(u_resid.shape[1]):
      u_resid_i = u_resid[:, i]
      real_col = np.zeros_like(u_resid_i)

      p_u = self.tail_fit.p_u
      p_l = self.tail_fit.p_l

      u_upper = self.tail_fit.u_upper
      u_lower = self.tail_fit.u_lower

      lower_mask = u_resid_i < p_l
      upper_mask = u_resid_i > (1 - p_u)
      body_mask = (~lower_mask) & (~upper_mask)

      if np.any(upper_mask):
        ppf_u = (
            self.tail_fit.u_upper
            + genpareto.ppf(
                (u_resid_i[upper_mask] - (1-p_u)) / p_u,
                self.tail_fit.c_U,
                scale=self.tail_fit.scale_U
            )
        )
        real_col[upper_mask] = ppf_u

      if np.any(body_mask):
        body = (u_resid_i[body_mask] - p_l) / (1 - p_l - p_u) * 100
        body_percentiles = np.clip(body, 0, 100)
        ppf_b = np.percentile(self.tail_fit.ecdf_data, body_percentiles)
        real_col[body_mask] = ppf_b

      if np.any(lower_mask):
        ppf_l = (
            self.tail_fit.u_lower
            - genpareto.ppf(
                1 - u_resid_i[lower_mask]/p_l,
                self.tail_fit.c_L,
                scale=self.tail_fit.scale_L
            )
        )
        real_col[lower_mask] = ppf_l

      z_resid_final[:, i] = real_col

    return z_resid_final


In [217]:
class GARCHEVTCOPULA:
  def __init__(self, sim_params, debug, **kwargs):
    self.debug = debug
    self.model_p = sim_params
    self.model_fit = GARCHEVTCOPULAFit()

  def _fit_model(self, r, ticker=None):
    best_bic = np.inf
    best_model = best_fit = best_lags = None

    for lags in range(1, self.model_p.lags + 1):
      model = arch_model(r, mean="AR", lags=lags, vol="GARCH", p=1, o=1, q=1, dist='studentst')
      fit = model.fit(disp='off', show_warning=False)
      if fit.convergence_flag:
        continue

      alpha, gamma, beta = fit.params["alpha[1]"], fit.params["gamma[1]"], fit.params["beta[1]"]
      if alpha + beta + gamma * 0.5 < 1 and fit.bic < best_bic:
        best_bic = fit.bic
        best_model, best_fit, best_lags = model, fit, lags

    if best_model is None:
      for lags in range(1, self.model_p.lags + 1):
        model = arch_model(r, mean="AR", lags=lags, vol="GARCH", p=1, o=0, q=1, dist='studentst')
        fit = model.fit(disp='off', show_warning=False)
        if fit.convergence_flag:
          continue
        alpha, beta = fit.params["alpha[1]"], fit.params["beta[1]"]
        print(alpha + beta)
        if alpha + beta < 1 and fit.bic < best_bic:
          best_bic = fit.bic
          best_model, best_fit, best_lags = model, fit, lags

    if best_model is None:
      raise ValueError(f"No stationary GARCH fit found")

    return best_model, best_fit, best_lags


  def _fit_arma_garch(self, returns):
    if self.debug:
      print("------------- Fitting AR-GARCH -------------")

    log_returns = np.log1p(returns)
    filterred_resid = pd.DataFrame(index=returns.index, columns=returns.columns)
    cols = returns.columns

    for i, col in enumerate(cols):
      if self.debug:
        print(f"Fitting {col} | {i+1}/{len(cols)}")

      r_scaled = log_returns[col] * self.model_p.scale_factor
      self.model_fit.vol_init[col] = r_scaled.std()
      self.model_fit.r_init[col] = r_scaled.mean()

      best_model, best_fit, best_lags = self._fit_model(r_scaled)
      self.model_fit.best_models[col] = best_model
      self.model_fit.best_fits[col] = best_fit
      self.model_fit.best_lags[col] = best_lags

      filterred_resid[col] = best_fit.std_resid

    return filterred_resid

  def _get_uniform_residuals(self, residuals):
    if self.debug:
      print("------------ Uniform Residuals -------------")

    u_resid = pd.DataFrame(index=residuals.index, columns=residuals.columns)

    for ticker in residuals.columns:
      model = MarginalTailModel(self.model_p, ticker)
      u_resid_i = model.fit_dist(residuals[ticker])
      u_resid[ticker] = u_resid_i
      self.model_fit.marginal_distributions[ticker] = model

    return u_resid

  def _inverse_semi_parametric_cdf(self, uniform_samples):
    if self.debug:
      print("-------- Inverse Semi Parametric CDF -------")

    n_paths, n_steps, n_assets = uniform_samples.shape
    z_resid = np.zeros((n_paths, n_steps, n_assets), dtype=float)

    eps = 1e-6

    for asset_idx, ticker in enumerate(self.cols):
      model = self.model_fit.marginal_distributions[ticker]
      u_slice = uniform_samples[:, :, asset_idx].T
      u_slice = np.clip(u_slice, eps, 1-eps)
      z_slice = model.inverse_cdf(u_slice)
      z_resid[:, :, asset_idx] = z_slice.T

    return z_resid

  def _fit_vine_copula(self, u_resid):
    if self.debug:
      print("----------- Fitting Vine Copula ------------")

    np_resid = u_resid.to_numpy()
    controls = pv.FitControlsVinecop()
    self.copula = pv.Vinecop.from_data(np_resid, controls=controls)


  def fit(self, returns):
    z_resid = self._fit_arma_garch(returns)
    self.cols = z_resid.columns
    self.start = z_resid.index[0]
    self.end = z_resid.index[-1]

    u_resid = self._get_uniform_residuals(z_resid)

    self._fit_vine_copula(u_resid)

  def _convert_to_returns(
    self,
    sim_r,
    model,
    params,
    ticker
  ):
    if self.debug:
      print(f"------ Converting {ticker} to Returns -----")

    lags = model.lags
    const = params["Const"]
    ar = params.get(f"{ticker}{lags[0]}", 0.0)
    ma = params.get(f"{ticker}{lags[1]}", 0.0)
    omega = params["omega"]
    alpha = params["alpha[1]"]
    gamma = params["gamma[1]"]
    beta = params["beta[1]"]
    nu = params["nu"]

    r_t = np.zeros_like(sim_r)

    sigma2 = np.zeros_like(sim_r)
    sigma2_prev = self.model_fit.vol_init[ticker]**2
    r_prev = self.model_fit.r_init[ticker]
    eps_prev = np.sqrt(sigma2_prev) * sim_r[:, 0]

    for t in range(sim_r.shape[-1]):
      I = np.where(eps_prev < 0, 1.0, 0.0)
      sigma2[:, t] = (
          omega
          + (alpha + gamma * I) * eps_prev**2
          + beta * sigma2_prev
      )

      sigma2_prev = sigma2[:, t]

      eps_t = np.sqrt(sigma2[:, t]) * sim_r[:, t]

      ar_term = np.sum(ar*r_prev) if isinstance(ar, np.ndarray) else ar*r_prev
      ma_term = np.sum(ma*eps_t) if isinstance(ma, np.ndarray) else ma*eps_t

      mu_t = const + ar_term + ma_term
      r_t[:, t] = mu_t + eps_t

      r_prev = r_t[:, t]
      eps_prev = eps_t

    return r_t


  def generate_sample(self):
    if self.debug:
      print("------------ Generating Samples ------------")

    n_paths = self.model_p.n_paths
    n_steps = self.model_p.n_steps
    n_assets = len(self.cols)

    total_obs = n_paths * n_steps
    simulated_steps = self.copula.simulate(n=total_obs)
    simulated_uniforms = simulated_steps.reshape(n_paths, n_steps, n_assets)

    sim_resid = self._inverse_semi_parametric_cdf(simulated_uniforms)
    sim_resid_T = np.transpose(sim_resid, (0, 2, 1))

    sim_r_log = np.zeros_like(sim_resid_T)
    for i, ticker in enumerate(self.cols):
      params = self.model_fit.best_fits[ticker].params
      model = self.model_fit.best_models[ticker]

      log_paths = self._convert_to_returns(
        sim_resid_T[:, i, :],
        model,
        params,
        ticker
      ) / self.model_p.scale_factor

      sim_r_log[:, i, :] = log_paths

    sim_r = np.expm1(sim_r_log)
    return sim_r


In [218]:
class CVaR(GARCHEVTCOPULA):
  def __init__(
      self,
      sim_params:SimParams,
      params:ModelParams,
      debug:bool=False,
      **kwargs
  ):
    super().__init__(
      debug=debug,
      params=params,
      sim_params=sim_params,
      **kwargs
    )
    self.debug = debug
    self.cvar_p = params
    self.sim_path = None

  def fit_garch_evt_copula(self, returns):
    self.start = returns.index[0]
    self.end = returns.index[-1]
    self.universe = returns.columns

    self.sim_path = f"simulation_{self.start}_{self.end}.npy"
    if not os.path.exists(self.sim_path):
      self.fit(returns)

  def generate_r_sample(self):
    if self.sim_path is None:
      raise ValueError("Fit the model first!")

    if os.path.exists(self.sim_path):
      sim_returns = np.load(self.sim_path)
    else:
      sim_returns = self.generate_sample()
      np.save(self.sim_path, sim_returns)

    return sim_returns

  def verify_eq_cvar(self, sim_returns, w=None):
    if w is None:
      w = np.ones(len(self.universe)) / len(self.universe)

    asset_R = np.prod(1 + sim_returns, axis=2) - 1
    portfolio_R = asset_R @ w

    portfolio_L = -portfolio_R
    asset_L = -asset_R

    alpha = self.cvar_p.cvar_q * 100
    var_p = np.percentile(portfolio_L, alpha)

    tail_mask = portfolio_L >= var_p

    tail_losses = asset_L[tail_mask]

    cvar_p = np.mean(portfolio_L[tail_mask])
    mcvar = np.mean(tail_losses, axis=0)

    euler_cvar_p = np.sum(w * mcvar)
    print(f"Direct Portfolio CVaR: {cvar_p:.6f}")
    print(f"Euler Sum MCVaR:       {euler_cvar_p:.6f}")

    assert np.isclose(cvar_p, euler_cvar_p, atol=1e-8), "Euler decomposition failed!"

    return mcvar, cvar_p

In [ ]:
class TailWeightedKendall:
  def __init__(
    self,
    params,
    **kwargs
  ):
    super().__init__(params=params, **kwargs)
    if not 0.0 < params.tail_threshold < 1.0:
      raise ValueError(
        "tail_threshold must lie between 0 and 1."
      )

    if params.tail_power <= 0:
      raise ValueError(
        "tail_power must be positive."
      )

    self.tail_threshold = params.tail_threshold
    self.tail_power = params.tail_power

    self.min_effective_sample = params.min_effective_sample

    self.shrink = params.shrink
    self.shrink_target_ess = params.shrink_target_ess

  @staticmethod
  def compound_returns(sim_returns):
      sim_returns = np.asarray(
        sim_returns,
        dtype=float
      )

      if sim_returns.ndim != 3:
        raise ValueError(
          "sim_returns must have shape "
          "(n_paths, n_assets, n_steps)."
        )

      return (
          np.prod(1.0 + sim_returns, axis=2) - 1.0
      )

  @staticmethod
  def empirical_uniforms(losses):
      losses = np.asarray(
        losses,
        dtype=float
      )

      if losses.ndim != 2:
        raise ValueError(
          "losses must be two-dimensional."
        )

      n_paths, n_assets = losses.shape

      U = np.empty_like(
        losses,
        dtype=float
      )

      for i in range(n_assets):
        U[:, i] = (
          rankdata(
            losses[:, i],
            method="average"
          ) / (n_paths + 1.0)
        )

      return U

  def tail_weight(self, u):
      u = np.asarray(u, dtype=float)

      z = np.maximum(
        (u - self.tail_threshold) / (1.0 - self.tail_threshold),
        0.0
      )

      return z ** self.tail_power

  @staticmethod
  def effective_sample_size(weights):
    weights = np.asarray(weights, dtype=float)

    s1 = weights.sum()
    s2 = np.sum(weights ** 2)

    if s2 <= 0:
        return 0.0

    return (s1 ** 2) / s2

  @staticmethod
  def _weighted_kendall(x, y, obs_weights):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(obs_weights, dtype=float)

    n = len(x)
    if n < 2:
      return np.nan

    numerator = 0.0
    denominator = 0.0

    for p in range(n - 1):
      dx = x[p] - x[p + 1:]
      dy = y[p] - y[p + 1:]

      pair_weights = (w[p] * w[p + 1:])

      signs = np.sign(dx * dy)

      numerator += np.sum(pair_weights * signs)

      denominator += np.sum(pair_weights)

    if denominator <= 0:
      return np.nan

    return numerator / denominator

  def pair_tau(self, losses_i, losses_j, U_i, U_j):
    wi = self.tail_weight(U_i)
    wj = self.tail_weight(U_j)
    pair_weights = wi * wj
    mask = pair_weights > 0
    n_tail = int(mask.sum())

    tau_global, _ = kendalltau(losses_i, losses_j)
    if not np.isfinite(tau_global):
        tau_global = 0.0

    if n_tail < 2:
        return {
            "tau_tail": np.nan,
            "tau_global": tau_global,
            "tau_final": np.clip(tau_global, -1.0, 1.0),
            "ess": 0.0,
            "n_tail": n_tail,
        }

    x = losses_i[mask]
    y = losses_j[mask]
    w = pair_weights[mask]

    ess = self.effective_sample_size(w)
    tau_tail = self._weighted_kendall(x, y, w)

    if self.shrink and np.isfinite(tau_tail):
        gamma = np.clip(ess / self.shrink_target_ess, 0.0, 1.0)
        tau_final = gamma * tau_tail + (1.0 - gamma) * tau_global
    else:
        tau_final = tau_tail if np.isfinite(tau_tail) else tau_global

    if not np.isfinite(tau_final):
        tau_final = 0.0

    tau_final = np.clip(tau_final, -1.0, 1.0)

    return {
        "tau_tail": tau_tail,
        "tau_global": tau_global,
        "tau_final": tau_final,
        "ess": ess,
        "n_tail": n_tail,
    }

  def fit(
    self,
    sim_returns,
    assets=None,
  ):
    horizon_returns = self.compound_returns(sim_returns)

    losses = -horizon_returns

    n_paths, n_assets = losses.shape

    if assets is None:
      assets = [
        f"asset_{i}"
        for i in range(n_assets)
      ]

    if len(assets) != n_assets:
      raise ValueError(
        "len(assets) must equal "
        "sim_returns.shape[1]."
      )

    U = self.empirical_uniforms(
        losses
    )

    tau = np.eye(n_assets, dtype=float)
    tau_tail = np.eye(n_assets, dtype=float)
    tau_global = np.eye(n_assets, dtype=float)
    ess = np.zeros((n_assets, n_assets), dtype=float)
    n_tail = np.zeros((n_assets, n_assets), dtype=int)

    for i in range(n_assets):
      ess[i, i] = n_paths
      n_tail[i, i] = n_paths

      for j in range(i + 1, n_assets):
        result = self.pair_tau(losses[:, i], losses[:, j], U[:, i], U[:, j])

        tau[i, j] = result[ "tau_final"]
        tau[j, i] = tau[i, j]
        tau_tail[i, j] = result["tau_tail"]
        tau_tail[j, i] = tau_tail[i, j]
        tau_global[i, j] = result["tau_global"]
        tau_global[j, i] = tau_global[i, j]

        ess[i, j] = result["ess"]
        ess[j, i] = ess[i, j]

        n_tail[i, j] = result[ "n_tail"]
        n_tail[j, i] = n_tail[i, j]

    distance = np.sqrt(
      0.5 * (1.0 - np.clip(tau, -1.0, 1.0))
    )

    np.fill_diagonal(distance, 0.0)

    return pd.DataFrame(distance, index=assets, columns=assets)


  def get_tau_dist(self, sim_returns, assets=None):
    return self.fit(sim_returns, assets)

In [229]:
class ClusterEngine(TailWeightedKendall):
  def __init__(self, params: ModelParams, debug=False, **kwargs):
    super().__init__(
        params=params,
        debug=debug,
        **kwargs
    )
    self.cp = ClusterParams()
    self.p = params


  def _get_tail_mask(self, sim_returns, assets, w=None):
    asset_R = np.prod(
        1.0 + sim_returns,
        axis=2
    ) - 1.0

    asset_L = -asset_R

    n_assets = asset_L.shape[1]

    if w is None:
        w = np.ones(n_assets) / n_assets

    portfolio_L = asset_L @ w

    var_p = np.percentile(
      portfolio_L,
      self.p.cvar_q*100
    )

    tail_mask = portfolio_L >= var_p

    return tail_mask


  def get_dist_mtx(self, returns:np.ndarray|pd.DataFrame, assets=None):
    if self.p.distance_method == "kendalltau":
      assert assets is not None, "Assets must be provided for Kendall Tau"
      dist_mtx = self.get_tau_dist(returns, assets)

    elif self.p.distance_method == "pearsoncorr":
      corr_mtx = np.clip(np.corrcoef(returns, rowvar=False), -1.0, 1.0)
      dist_mtx = np.sqrt(0.5 * (1 - corr_mtx))

    return dist_mtx

  def _generate_null_dists(self, returns_df):
    N_samples, N_assets = returns_df.shape
    returns_arr = returns_df.values
    min_bounds = np.min(returns_arr, axis=0)
    max_bounds = np.max(returns_arr, axis=0)

    null_dists, null_links = [], []

    for i in range(self.p.n_sims):
      null_returns = np.random.uniform(
        low=min_bounds,
        high=max_bounds,
        size=(self.p.n_sims, N_samples, N_assets)
      )

      null_dist = self.get_dist_mtx(
        np.transpose(null_returns, (0, 2, 1)),
        returns_df.columns
      )

      condensed_dist = squareform(null_dist, checks=False)
      Z_null = linkage(condensed_dist, method="single")

      null_dists.append(null_dist)
      null_links.append(Z_null)

    return np.array(null_dists), np.array(null_links)

  def _compute_cl_dispersion(self, dist_mtx, cluster_labels):
    if isinstance(dist_mtx, pd.DataFrame):
      dist_mtx = dist_mtx.to_numpy()

    unique_clusters = np.unique(cluster_labels)
    W_k = 0.0

    for c_id in unique_clusters:
      cluster_indices = np.where(cluster_labels == c_id)[0]
      cluster_dist = dist_mtx[np.ix_(cluster_indices, cluster_indices)]

      norm = 2*len(cluster_indices)

      D_r = np.sum(cluster_dist**2)

      W_k += D_r / norm

    return W_k

  def _compute_ref_log_disp(self, null_dists, null_links, k):
    B = self.p.n_sims
    W_k_log = []

    for b in range(B):
      clusters = fcluster(null_links[b], t=k, criterion="maxclust")

      W_k = self._compute_cl_dispersion(null_dists[b], clusters)
      W_k_log.append(np.log(max(W_k, 1e-300)))

    W_k_log = np.array(W_k_log)
    E_W_k = np.mean(W_k_log)

    sdk = np.std(W_k_log, ddof=1) if B > 1 else 0.0
    s_k = sdk * np.sqrt(1+1/B)

    return E_W_k, s_k

  def _get_k_clusters(self, dist_mtx, Z, returns):
    if self.debug:
      print("---------- Calculating K Clusters ----------")

    k_max = min(self.p.k_max, dist_mtx.shape[0]-1)
    Gap_k, s_k_list = [], []
    null_dists, null_links = self._generate_null_dists(returns)
    k_range = list(range(1, k_max+1))

    for k in k_range:
      clusters = fcluster(Z, t=k, criterion="maxclust")

      W_k_real = self._compute_cl_dispersion(dist_mtx, clusters)
      log_disp_k = np.log(max(W_k_real, 1e-300))

      E_W_k, s_k = self._compute_ref_log_disp(null_dists, null_links, k)

      Gap_k.append(E_W_k - log_disp_k)
      s_k_list.append(s_k)

    gaps = np.array(Gap_k)
    s_k_list = np.array(s_k_list)

    optimal_k = k_range[np.argmax(gaps)]
    for idx in range(len(k_range)-1):
      if gaps[idx] >= gaps[idx+1] - s_k_list[idx+1]:
        optimal_k = k_range[idx]
        break

    optimal_k = max(optimal_k, 2) if dist_mtx.shape[0] > 1 else optimal_k
    if self.debug:
      print(f"Optimal k: {optimal_k}")
    return int(optimal_k)


  def get_clusters(self, dist_mtx:pd.DataFrame, returns:pd.DataFrame) -> pd.DataFrame:
    comp_disp = squareform(dist_mtx, checks=False)
    Z = linkage(comp_disp, method="single")

    optimal_k = self._get_k_clusters(dist_mtx, Z, returns)
    labels = fcluster(Z, t=optimal_k, criterion="maxclust")

    return pd.DataFrame({'Asset': returns.columns, 'Cluster': labels}), Z


  def _order_cluster_ids(self, Z, clusters_df):
    label_by_asset = clusters_df.set_index("Asset")["Cluster"]
    leaf_order = leaves_list(Z)

    asset_order = clusters_df["Asset"].values[leaf_order]
    ordered_labels = label_by_asset.loc[asset_order].values

    seen, ordered_ids = set(), []
    for lbl in ordered_labels:
      if lbl not in seen:
        seen.add(lbl)
        ordered_ids.append(lbl)

    return ordered_ids

In [230]:
class Optimizer:
  def __init__(self, params, sim_params, debug=False, **kwargs):
    super().__init__(
        params=params,
        sim_params=sim_params,
        debug=debug,
        **kwargs
    )
    self.p = params
    self.debug = debug
    self.lam = 1
    self.solver_chain = (cp.CLARABEL, cp.ECOS, cp.SCS)

  def _calculate_w(self, x):
    x_opt = np.asarray(x.value).ravel()
    w = x_opt/x_opt.sum()
    return w

  def _get_intra_cluster_w(
    self,
    cluster_paths:np.ndarray,
    cluster_idx:int,
    euler_risk_check_fn=None,
    risk_budget:np.ndarray=None
  ):
    if self.debug:
      print(f"--------- Fitting w for cluster: {cluster_idx} ---------")

    cluster_R_unscaled = np.prod(1 + cluster_paths, axis=2) - 1
    if not np.all(np.isfinite(cluster_R_unscaled)):
      raise ValueError(f"Non-finite returns in cluster {cluster_idx}: "
                      f"{np.isnan(cluster_R_unscaled).sum()} NaN, \
                      {np.isinf(cluster_R_unscaled).sum()} inf")

    cluster_R = cluster_R_unscaled * self.p.return_scale
    S, N = cluster_R.shape
    x = cp.Variable(N, pos=True)

    if risk_budget is None:
      b = np.ones(N)/N
    else:
      b = np.asarray(risk_budget, dtype=float)
      b = b/np.sum(b)

    zeta = cp.Variable()

    cluster_L = -(cluster_R@x)
    excess = cp.pos(cluster_L - zeta)

    cvar = zeta + cp.mean(excess)/(1-self.p.cvar_q)

    barrier = -self.lam*cp.sum(cp.multiply(b, cp.log(x)))

    eps = 1e-6
    reg = eps * cp.sum(x)

    obj = cp.Minimize(cvar + reg + barrier)

    constraints = [
      x <= 10
    ]

    prob = cp.Problem(obj, constraints)
    for solver in self.solver_chain:
      try:
        prob.solve(solver=solver)
        if prob.status in ("optimal", "optimal_inaccurate"):
          print(solver)
          break
      except cp.error.SolverError:
        continue
    else:
      raise ValueError(f"All solvers failed for cluster {cluster_idx}: status={prob.status}")

    if np.any(np.isclose(x.value, 10, atol=1e-4)):
      print(
          f"Cluster {cluster_idx}: box constraint binding — risk parity not exact"
      )

    w = self._calculate_w(x)

    if euler_risk_check_fn is not None:
      euler_risk_check_fn(cluster_paths, w)

    return w, cluster_R_unscaled



In [231]:
class CVaRHERCPortfolio(DataStore, ClusterEngine, Optimizer, CVaR):
  def __init__(self, params, sim_params, debug=False, **kwargs):
    super().__init__(
        params=params,
        sim_params=sim_params,
        debug=debug,
        **kwargs
    )
    self.p = params
    self.debug = debug
    self.sim_p = sim_params

  def get_data(self, universe, start, end, benchmark="^GSPC"):
    returns, benchmark = self._get_data(
      universe=universe,
      start=start,
      end=end,
      benchmark=benchmark
    )

    return returns, benchmark

  def _recursive_bisect(self, ordered_ids, cluster_return_series):
    weights = {cid: 1.0 for cid in ordered_ids}

    def node_risk(ids):
      if len(ids) == 1:
        r = cluster_return_series[ids[0]]
        var = np.quantile(-r, self.p.cvar_q)
        tail = -r[-r >= var]
        cvar = tail.mean() if len(tail) else var

        return max(cvar, 1e-6)

      member_R = np.stack([cluster_return_series[c] for c in ids], axis=1)
      node_w = np.ones(len(ids)) / len(ids)
      portfolio = member_R @ node_w

      portfolio_L = -portfolio
      var = np.quantile(portfolio_L, self.p.cvar_q)
      tail = portfolio_L[portfolio_L >= var]
      cvar = np.mean(tail) if len(tail) else var

      return max(cvar, 1e-6)

    def bisect(ids):
      if len(ids) <= 1:
        return

      mid = len(ids) // 2
      left, right = ids[:mid], ids[mid:]
      r_l, r_r = node_risk(left), node_risk(right)
      alpha = 1 - r_l / (r_l + r_r)

      for c in left:  weights[c] *= alpha
      for c in right: weights[c] *= (1 - alpha)

      bisect(left)
      bisect(right)

    bisect(ordered_ids)
    return weights

  def get_w(self, returns):
    self.fit_garch_evt_copula(returns)
    sim_returns = self.generate_r_sample()

    dist_mtx = self.get_dist_mtx(sim_returns, returns.columns)
    self.dist_mtx = dist_mtx

    clusters_df, Z = self.get_clusters(dist_mtx, returns)
    self.clusters = clusters_df

    unique = np.unique(clusters_df.Cluster.values)


    w_by_cluster = {}
    cluster_return_series = {}

    for c_id in unique:
      cols_i = clusters_df.Cluster == c_id
      sim_returns_i = sim_returns[:, cols_i]
      if sim_returns_i.shape[1] > 1:
        w_i, _ = self._get_intra_cluster_w(
            cluster_paths=sim_returns_i,
            cluster_idx=c_id,
            euler_risk_check_fn=self.verify_eq_cvar
        )
      else:
        w_i = np.ones(1)

      w_by_cluster[c_id] = w_i

      cluster_R_i = np.prod(1 + sim_returns_i, axis=2) - 1
      cluster_return_series[c_id] = cluster_R_i @ w_i

    ordered_ids = self._order_cluster_ids(Z, clusters_df)
    cluster_weights = self._recursive_bisect(ordered_ids, cluster_return_series)

    w_final = []
    for c_id, w_i in w_by_cluster.items():
      w_final.append(w_i * cluster_weights[c_id])

    w_df = np.concatenate(w_final)
    return w_df


